# 06 – Debug mạng nơ-ron

Notebook kiểm tra dữ liệu, forward pass, loss, activation và gradient trước khi chạy huấn luyện dài. Mặc định dùng SimpleCNN và một batch nhỏ; không tải pretrained weights và không ghi đè checkpoint.

Chạy lần lượt từ trên xuống. Nếu một `assert` thất bại, xử lý lỗi ở bước đó trước khi tiếp tục.

## 1. Thiết lập và nạp một batch

Đổi `DEBUG_MODEL` sang `resnet18` hoặc `efficientnet_b0` nếu cần kiểm tra kiến trúc đang dùng trong thí nghiệm. Các mô hình này được khởi tạo không pretrained để tránh tải weights qua mạng.

In [1]:
from pathlib import Path
import sys
import torch
import torch.nn as nn
import pandas as pd
from PIL import Image

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.data import make_dataloaders
from src.model import SimpleCNN, build_resnet18, build_efficientnet_b0

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEBUG_MODEL = 'simplecnn'  # 'simplecnn', 'resnet18', hoặc 'efficientnet_b0'
DEBUG_BATCH_SIZE = 4
metadata = pd.read_csv(PROJECT_ROOT / 'data' / 'metadata.csv')
NUM_CLASSES = int(metadata['label'].nunique())
loaders = make_dataloaders(metadata, batch_size=DEBUG_BATCH_SIZE, augmentation=False)
images, labels = next(iter(loaders['train']))

print(f'Device: {DEVICE} | Model: {DEBUG_MODEL} | Classes: {NUM_CLASSES}')
print(f'Batch shape: {tuple(images.shape)} | dtype: {images.dtype}')
print(f'Labels shape: {tuple(labels.shape)} | dtype: {labels.dtype}')

Device: cpu | Model: simplecnn | Classes: 38
Batch shape: (4, 3, 224, 224) | dtype: torch.float32
Labels shape: (4,) | dtype: torch.int64


## 2. Kiểm tra metadata, ảnh và batch

Tìm đường dẫn ảnh thiếu, kiểm tra một mẫu ảnh có thể giải mã, rồi xác nhận tensor và nhãn có shape, kiểu dữ liệu, miền giá trị hợp lệ.

In [2]:
required_columns = {'path', 'label', 'split'}
assert required_columns.issubset(metadata.columns), f'Thiếu cột: {required_columns - set(metadata.columns)}'
assert metadata['label'].notna().all(), 'Nhãn có giá trị rỗng'
assert metadata['split'].isin(['train', 'val', 'test']).all(), 'Giá trị split không hợp lệ'

missing_paths = [path for path in metadata['path'] if not Path(path).is_file()]
print(f'Ảnh không tồn tại: {len(missing_paths)} / {len(metadata)}')
if missing_paths:
    print('Ví dụ đường dẫn thiếu:', missing_paths[:5])

sample_path = Path(metadata.iloc[0]['path'])
with Image.open(sample_path) as image:
    image.verify()
print(f'Ảnh mẫu đọc được: {sample_path.name}')

assert images.ndim == 4 and images.shape[1] == 3, f'Shape ảnh không hợp lệ: {tuple(images.shape)}'
assert images.dtype == torch.float32, f'Dtype ảnh cần float32, nhận {images.dtype}'
assert torch.isfinite(images).all(), 'Batch có NaN hoặc Inf'
assert labels.ndim == 1 and labels.dtype == torch.int64, 'Nhãn cần là tensor int64 một chiều'
assert labels.min() >= 0 and labels.max() < NUM_CLASSES, 'Nhãn nằm ngoài [0, NUM_CLASSES)'
print(f'Batch hợp lệ | pixel sau chuẩn hóa: [{images.min():.3f}, {images.max():.3f}] | labels: {labels.tolist()}')
print('Số mẫu theo split:\n', metadata['split'].value_counts().to_string())

Ảnh không tồn tại: 0 / 54305
Ảnh mẫu đọc được: 01a66316-0e98-4d3b-a56f-d78752cd043f___FREC_Scab 3003.JPG
Batch hợp lệ | pixel sau chuẩn hóa: [-1.981, 2.640] | labels: [16, 9, 24, 15]
Số mẫu theo split:
 split
train    38013
test      8146
val       8146


## 3. Forward, loss, activation và gradient

Hooks chỉ thu thống kê ở Conv2d, BatchNorm2d và Linear trong một forward pass. Bảng gradient giúp phát hiện tham số không nhận gradient hoặc gradient không hữu hạn.

In [3]:
def build_debug_model(name: str, num_classes: int) -> nn.Module:
    if name == 'simplecnn':
        return SimpleCNN(num_classes)
    if name == 'resnet18':
        return build_resnet18(num_classes, pretrained=False, freeze_backbone=False)
    if name == 'efficientnet_b0':
        return build_efficientnet_b0(num_classes, pretrained=False, freeze_backbone=False)
    raise ValueError(f'DEBUG_MODEL không được hỗ trợ: {name}')

model = build_debug_model(DEBUG_MODEL, NUM_CLASSES).to(DEVICE)
images_device, labels_device = images.to(DEVICE), labels.to(DEVICE)
activation_stats = []
handles = []

def record_activation(name):
    def hook(_module, _inputs, output):
        if isinstance(output, torch.Tensor):
            values = output.detach().float()
            activation_stats.append({
                'layer': name,
                'shape': tuple(values.shape),
                'finite': bool(torch.isfinite(values).all().item()),
                'mean': values.mean().item(),
                'std': values.std(unbiased=False).item(),
            })
    return hook

for name, layer in model.named_modules():
    if isinstance(layer, (nn.Conv2d, nn.BatchNorm2d, nn.Linear)):
        handles.append(layer.register_forward_hook(record_activation(name or '<root>')))

model.train()
model.zero_grad(set_to_none=True)
logits = model(images_device)
for handle in handles:
    handle.remove()

assert logits.shape == (len(labels), NUM_CLASSES), f'Shape logits sai: {tuple(logits.shape)}'
assert torch.isfinite(logits).all(), 'Logits có NaN hoặc Inf'
loss = nn.CrossEntropyLoss()(logits, labels_device)
assert torch.isfinite(loss), f'Loss không hữu hạn: {loss.item()}'
loss.backward()

print(f'Logits: {tuple(logits.shape)} | loss: {loss.item():.5f}')
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
print('Activation stats:')
display(pd.DataFrame(activation_stats))

gradient_rows = []
for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        if parameter.grad is None:
            gradient_rows.append({'parameter': name, 'grad_norm': None, 'finite': False, 'status': 'missing'})
        else:
            grad = parameter.grad.detach()
            norm = grad.norm().item()
            gradient_rows.append({'parameter': name, 'grad_norm': norm, 'finite': bool(torch.isfinite(grad).all().item()), 'status': 'zero' if norm == 0 else 'ok'})
gradient_df = pd.DataFrame(gradient_rows)
display(gradient_df)
print('Tham số thiếu gradient:', int((gradient_df['status'] == 'missing').sum()))
print('Gradient không hữu hạn:', int((~gradient_df['finite']).sum()))

Logits: (4, 38) | loss: 3.85234
Trainable parameters: 98,598
Activation stats:


,layer,shape,finite,mean,std
0,features.0,"(4, 32, 224, 224)",True,9.849020e-03,0.441998
1,features.1,"(4, 32, 224, 224)",True,-2.432843e-09,0.999923
2,features.4,"(4, 64, 112, 112)",True,-3.064784e-02,0.503921
3,features.5,"(4, 64, 112, 112)",True,-6.082107e-09,0.999951
4,features.8,"(4, 128, 56, 56)",True,4.121879e-03,0.503380
5,features.9,"(4, 128, 56, 56)",True,7.906738e-09,0.999957
6,classifier,"(4, 38)",True,9.806681e-04,0.232226


,parameter,grad_norm,finite,status
0,features.0.weight,1.440791e+00,True,ok
1,features.0.bias,4.131940e-05,True,ok
2,features.1.weight,9.488750e-02,True,ok
3,features.1.bias,6.070301e-02,True,ok
4,features.4.weight,2.789755e+00,True,ok
5,features.4.bias,8.309402e-07,True,ok
6,features.5.weight,7.215659e-02,True,ok
7,features.5.bias,5.036734e-02,True,ok
8,features.8.weight,3.224222e+00,True,ok
9,features.8.bias,9.043684e-07,True,ok


Tham số thiếu gradient: 0
Gradient không hữu hạn: 0


## 4. Kiểm tra khả năng học bằng cách overfit một batch

Đây là phép thử nhanh để phân biệt lỗi pipeline/model với vấn đề tổng quát hóa. Mô hình mới chỉ học lặp lại cùng một batch trong số bước giới hạn; không lưu checkpoint. Loss cuối cần thấp hơn loss đầu đáng kể.

In [4]:
overfit_model = build_debug_model(DEBUG_MODEL, NUM_CLASSES).to(DEVICE)
optimizer = torch.optim.AdamW(
    (parameter for parameter in overfit_model.parameters() if parameter.requires_grad),
    lr=1e-3,
    weight_decay=0.0,
)
criterion = nn.CrossEntropyLoss()
loss_trace = []

for step in range(20):
    overfit_model.train()
    optimizer.zero_grad(set_to_none=True)
    batch_logits = overfit_model(images_device)
    batch_loss = criterion(batch_logits, labels_device)
    if not torch.isfinite(batch_loss):
        raise FloatingPointError(f'Loss không hữu hạn tại bước {step}: {batch_loss.item()}')
    batch_loss.backward()
    optimizer.step()
    loss_trace.append(batch_loss.item())

print(f'Loss đầu: {loss_trace[0]:.5f} | loss cuối: {loss_trace[-1]:.5f}')
print(f'Tỷ lệ dự đoán đúng trên batch: {(overfit_model(images_device).argmax(1) == labels_device).float().mean().item():.1%}')
assert loss_trace[-1] < loss_trace[0], 'Loss không giảm: kiểm tra nhãn, learning rate, gradient và kiến trúc.'

Loss đầu: 3.55379 | loss cuối: 0.34821
Tỷ lệ dự đoán đúng trên batch: 100.0%


## 5. Gợi ý khoanh vùng

- Ảnh thiếu hoặc không giải mã được: kiểm tra `metadata.csv` và đường dẫn dữ liệu.
- Batch có NaN/Inf hoặc nhãn ngoài phạm vi: kiểm tra preprocessing và mapping nhãn.
- Logits sai shape hoặc không hữu hạn: kiểm tra số lớp đầu ra, input và forward của model.
- Gradient `missing`: tham số có thể bị tách khỏi loss hoặc bị freeze ngoài ý muốn; gradient bằng 0 liên tục cũng cần xem lại activation và graph.
- Loss không giảm khi overfit batch: kiểm tra nhãn, `requires_grad`, optimizer/learning rate và phép tính loss.
- Nếu các kiểm tra này qua nhưng validation vẫn kém, xem class imbalance, augmentation, split và learning curves trong các notebook 02–05.